# 04 - Stage 1: candidate misconception codes

`Qwen/Qwen3.8-27B` (thinking, effort `medium`, up to 49,152 new tokens) looks at the student photo, the question diagram, the question, the reference working, the pass-1 text, the pass-3 text (or the pass-2 text where pass 3 gave nothing usable) and the pass-2 review notes, and lists candidate codes from the 75-code taxonomy with a short reason each. Prompt: `resources/prompts/stage1_candidates_template.txt`.

The reasoning text is stored: stage 2 uses it to list codes the model considered but did not list.

**Reads** `work/pass1`, `work/pass2`, `work/pass3`.  **Writes** `work/stage1/{records/,frozen_inputs/jobs/,results.csv}`.

**Default: smoke test on 5 rows.** Set `LIMIT = None` in the second cell for the full run.

In [ ]:
import os, sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs' / 'models.json').is_file())
os.chdir(REPO); sys.path.insert(0, str(REPO / 'src'))
from misc_pipeline import common, runner
LIMIT = 5         # SMOKE TEST: first 5 test rows only. Set LIMIT = None for the full 3,498 rows (finished rows are kept and skipped, so the full run resumes from the smoke test).
print('repo:', REPO, '| data:', common.DATA, '| work:', common.WORK)

In [ ]:
from misc_pipeline import stage1, stage_exports
jobs, metas = stage1.build_jobs(LIMIT)
print(len(jobs), 'rows | pass-3 text used:', sum(m['used_pass3'] for m in metas.values()), '| changed by pass 3:', sum(m['changed'] for m in metas.values()))
print(jobs[0]['prompt'][-1500:])

In [ ]:
summary = runner.run_stage('stage1', jobs, common.STAGE_DIRS['stage1'], stage1.parse_record, label_images=True)
print(summary)

In [ ]:
exp = stage_exports.export_stage1(jobs, metas); print(exp)

## Free the GPUs
The stage already closes its model when it ends. Run this cell after an interrupted run or an error, or before starting the next notebook in the same kernel.

In [ ]:
import gc
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()
        print('GPU memory in use (GB):', [round(torch.cuda.memory_allocated(i) / 1e9, 1) for i in range(torch.cuda.device_count())])
except ImportError:
    pass
# If memory is still held, restart the kernel (Kernel > Restart); vLLM worker processes then exit.

## Optional: delete the downloaded model
The weights stay in the Hugging Face cache (`HF_HOME`) and are re-downloaded on the next use. Set `DELETE_MODEL = True` to remove `Qwen/Qwen3.8-27B` from the cache now. Keep it if a later notebook uses the same model (notebooks 02 and 03 share one model, as do 04 and 05).

In [ ]:
DELETE_MODEL = False
MODEL = 'Qwen/Qwen3.8-27B'
if DELETE_MODEL:
    from huggingface_hub import scan_cache_dir
    info = scan_cache_dir()
    hashes = [r.commit_hash for repo in info.repos if repo.repo_id == MODEL for r in repo.revisions]
    if hashes:
        strategy = info.delete_revisions(*hashes); print('freeing %.1f GB' % (strategy.expected_freed_size / 1e9)); strategy.execute()
    else:
        print(MODEL, 'is not in the cache')
else:
    print('model kept in the cache')